# State-from-Observation Demo

This notebook is a protected-data-free toy walkthrough of the repository's central idea:
a clinical window is represented as an observation tensor with `value`, `mask`, and `delta` channels,
then mapped to a latent state with the same encoder used in the main pipeline.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch

from information_state.observation_data import compute_stay_deltas
from information_state.state_from_observation import StateFromObservationModel

In [ ]:
full_values = np.array(
    [
        [82.0, 68.0, 1.2],
        [85.0, 66.0, 1.3],
        [91.0, 64.0, 1.8],
        [95.0, 60.0, 2.4],
    ],
    dtype=np.float32,
)
dense_mask = np.array(
    [
        [1, 1, 1],
        [1, 1, 0],
        [1, 1, 1],
        [1, 1, 1],
    ],
    dtype=np.uint8,
)
thin_mask = np.array(
    [
        [1, 1, 1],
        [0, 0, 0],
        [1, 0, 1],
        [0, 1, 0],
    ],
    dtype=np.uint8,
)

def forward_fill(values, mask):
    filled = np.zeros_like(values)
    for j in range(values.shape[1]):
        last_value = 0.0
        seen = False
        for t in range(values.shape[0]):
            if mask[t, j]:
                last_value = values[t, j]
                seen = True
            filled[t, j] = last_value if seen else 0.0
    return filled

dense_triplet = np.stack(
    [full_values, dense_mask.astype(np.float32), np.log1p(compute_stay_deltas(dense_mask, bin_hours=1, delta_cap_hours=6))],
    axis=-1,
)
thin_triplet = np.stack(
    [forward_fill(full_values, thin_mask), thin_mask.astype(np.float32), np.log1p(compute_stay_deltas(thin_mask, bin_hours=1, delta_cap_hours=6))],
    axis=-1,
)

dense_triplet.shape, thin_triplet.shape

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(10, 5), constrained_layout=True)
for row, tensor, title in [(0, dense_triplet, 'Dense observation'), (1, thin_triplet, 'Thinned observation')]:
    axes[row, 0].imshow(tensor[..., 0], aspect='auto', cmap='viridis')
    axes[row, 0].set_title(f'{title}: value')
    axes[row, 1].imshow(tensor[..., 1], aspect='auto', cmap='gray_r', vmin=0, vmax=1)
    axes[row, 1].set_title(f'{title}: mask')
    axes[row, 2].imshow(tensor[..., 2], aspect='auto', cmap='magma')
    axes[row, 2].set_title(f'{title}: log(1 + delta)')
plt.show()

In [ ]:
torch.manual_seed(7)
model = StateFromObservationModel(
    num_variables=3,
    num_time_bins=4,
    d_model=16,
    num_heads=4,
    num_layers=1,
    projection_dim=8,
    dropout=0.0,
)

dense_state = model.encode(torch.from_numpy(dense_triplet[None]).float())
thin_state = model.encode(torch.from_numpy(thin_triplet[None]).float())
drift = torch.linalg.vector_norm(dense_state - thin_state, dim=1)
print('Dense latent state shape:', tuple(dense_state.shape))
print('Toy embedding drift:', float(drift.item()))

In the full pipeline, the encoder is trained contrastively on adjacent windows so that this latent state becomes clinically meaningful.
The real-data robustness script then measures how much that learned state moves when the observation process is thinned.